# 🕉️ Gitanjali v2 - High Speed GPU Ingestion
This notebook processes the Bhagavad Gita, Mahabharata, and Ramayana using a T4 GPU and uploads them to MongoDB Atlas.

In [ ]:
!pip install langchain langchain-community langchain-mongodb langchain-groq langchain-huggingface pymongo[srv] sentence-transformers pypdf python-dotenv streamlit tiktoken accelerate dnspython

In [ ]:
import os
from google.colab import userdata
from pymongo import MongoClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Setup Configuration
try:
    MONGODB_ATLAS_CLUSTER_URI = userdata.get('MONGODB_ATLAS_CLUSTER_URI')
    DB_NAME = "gitanjali_v2"
    COLLECTION_NAME = "wisdom_base"
    INDEX_NAME = "vector_index"
except:
    print("Please set MONGODB_ATLAS_CLUSTER_URI in the Colab Secrets (Key icon).")

# 2. Initialize MongoDB
client = MongoClient(MONGODB_ATLAS_CLUSTER_URI)
collection = client[DB_NAME][COLLECTION_NAME]

print("Cleaning up existing data...")
collection.delete_many({})

# 3. Process PDFs
pdfs = [
    "Bhagavad-Gita As It Is.pdf",
    "Mahabharata (Unabridged in English).pdf",
    "Ramayana.of.Valmiki.by.Hari.Prasad.Shastri.pdf"
]

all_docs = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

for pdf in pdfs:
    if os.path.exists(pdf):
        print(f"Processing {pdf}...")
        loader = PyPDFLoader(pdf)
        docs = loader.load()
        source_name = pdf.replace(".pdf", "")
        for doc in docs:
            doc.metadata["source"] = source_name
        chunks = text_splitter.split_documents(docs)
        all_docs.extend(chunks)
    else:
        print(f"Warning: {pdf} not found in /content/")

print(f"Total chunks prepared: {len(all_docs)}")

# 4. Initialize BGE-M3 on GPU
print("Initializing BGE-M3 on T4 GPU...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# 5. Batch Upload
batch_size = 500
print(f"Uploading to MongoDB Atlas in batches of {batch_size}...")

for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    if i == 0:
        vector_search = MongoDBAtlasVectorSearch.from_documents(
            documents=batch,
            embedding=embeddings,
            collection=collection,
            index_name=INDEX_NAME
        )
    else:
        vector_search.add_documents(documents=batch)
    print(f"Progress: {min(i + batch_size, len(all_docs))}/{len(all_docs)} uploaded.")

print("\n✓ Ingestion complete!")

Cleaning up existing data...
Processing Bhagavad-Gita As It Is.pdf...
Processing Mahabharata (Unabridged in English).pdf...
Processing Ramayana.of.Valmiki.by.Hari.Prasad.Shastri.pdf...
Total chunks prepared: 26554
Initializing BGE-M3 on T4 GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Uploading to MongoDB Atlas in batches of 500...
Progress: 500/26554 uploaded.
Progress: 1000/26554 uploaded.
Progress: 1500/26554 uploaded.
Progress: 2000/26554 uploaded.
Progress: 2500/26554 uploaded.
Progress: 3000/26554 uploaded.
Progress: 3500/26554 uploaded.
Progress: 4000/26554 uploaded.
Progress: 4500/26554 uploaded.
Progress: 5000/26554 uploaded.
Progress: 5500/26554 uploaded.
Progress: 6000/26554 uploaded.
Progress: 6500/26554 uploaded.
Progress: 7000/26554 uploaded.
Progress: 7500/26554 uploaded.
Progress: 8000/26554 uploaded.
Progress: 8500/26554 uploaded.
Progress: 9000/26554 uploaded.
Progress: 9500/26554 uploaded.
Progress: 10000/26554 uploaded.
Progress: 10500/26554 uploaded.
Progress: 11000/26554 uploaded.
Progress: 11500/26554 uploaded.
Progress: 12000/26554 uploaded.
Progress: 12500/26554 uploaded.
Progress: 13000/26554 uploaded.
Progress: 13500/26554 uploaded.
Progress: 14000/26554 uploaded.
Progress: 14500/26554 uploaded.
Progress: 15000/26554 uploaded.
Progress: 15